In [1]:
import pandas as pd
import numpy as np

# Load the merged playoff modeling dataset
model_data = pd.read_csv(
    "../data/nba_playoff_model_data_1986_2025.csv"
)

# Convert Date back to datetime
model_data["Date"] = pd.to_datetime(model_data["Date"])

print("Dataset loaded.")
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

model_data.head()

Dataset loaded.
Rows: 3143
Columns: 26


,Season,Season_End,Date,Away_Team,Away_PTS,Home_Team,Home_PTS,Home_Win,Home_W_PCT,Home_SRS,...,Away_ORtg,Away_DRtg,Away_NRtg,Away_Pace,W_PCT_Diff,SRS_Diff,ORtg_Diff,DRtg_Diff,NRtg_Diff,Pace_Diff
0,1985-86,1986,1986-04-17,Chicago Bulls,104,Boston Celtics,123,1,0.817073,9.06,...,108.6,112.4,-3.8,99.7,0.451220,12.18,3.2,-9.8,13.0,1.5
1,1985-86,1986,1986-04-17,Detroit Pistons,122,Atlanta Hawks,140,1,0.609756,2.59,...,109.0,107.9,1.1,104.2,0.048780,1.15,-1.1,-2.3,1.2,-4.2
2,1985-86,1986,1986-04-17,San Antonio Spurs,88,Los Angeles Lakers,135,1,0.756098,6.84,...,106.5,108.2,-1.7,103.8,0.329268,8.90,6.8,-2.4,9.2,-1.1
3,1985-86,1986,1986-04-17,Sacramento Kings,87,Houston Rockets,107,1,0.621951,2.10,...,106.2,109.2,-3.0,102.0,0.170732,5.29,3.9,-1.6,5.5,1.5
4,1985-86,1986,1986-04-18,Utah Jazz,93,Dallas Mavericks,101,1,0.536585,0.70,...,104.2,104.6,-0.4,103.3,0.024390,1.37,7.9,6.4,1.5,-1.1


In [2]:
# -----------------------------
# SIMPLE BASELINES
# -----------------------------

# Baseline A:
# Always predict that the home team wins
home_only_accuracy = model_data["Home_Win"].mean()

# Baseline B:
# Predict that whichever team had the better SRS wins
model_data["SRS_Prediction"] = (
    model_data["SRS_Diff"] > 0
).astype(int)

srs_accuracy = (
    model_data["SRS_Prediction"] ==
    model_data["Home_Win"]
).mean()

print("Always pick home team:")
print(f"Accuracy: {home_only_accuracy:.3f}")

print("\nPick team with better regular-season SRS:")
print(f"Accuracy: {srs_accuracy:.3f}")

Always pick home team:
Accuracy: 0.639

Pick team with better regular-season SRS:
Accuracy: 0.615


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss
)

print("scikit-learn loaded successfully.")

ModuleNotFoundError: No module named 'sklearn'

In [4]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 17.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss
)

print("scikit-learn loaded successfully.")

scikit-learn loaded successfully.


In [6]:
# Chronological train/test split

train_data = model_data[
    model_data["Season_End"] <= 2015
].copy()

test_data = model_data[
    model_data["Season_End"] >= 2016
].copy()

print("Training period:")
print(
    train_data["Season"].min(),
    "to",
    train_data["Season"].max()
)
print("Training games:", len(train_data))

print("\nTest period:")
print(
    test_data["Season"].min(),
    "to",
    test_data["Season"].max()
)
print("Test games:", len(test_data))

Training period:
1985-86 to 2014-15
Training games: 2309

Test period:
2015-16 to 2024-25
Test games: 834


In [7]:
# -----------------------------
# MODEL 1: SRS + HOME COURT
# -----------------------------

X_train = train_data[["SRS_Diff"]]
y_train = train_data["Home_Win"]

X_test = test_data[["SRS_Diff"]]
y_test = test_data["Home_Win"]

# Fit logistic regression
model_1 = LogisticRegression()
model_1.fit(X_train, y_train)

# Predicted probability that HOME team wins
model_1_prob = model_1.predict_proba(X_test)[:, 1]

# Convert probabilities into winner predictions
model_1_pred = (model_1_prob >= 0.5).astype(int)

# Evaluate
model_1_accuracy = accuracy_score(y_test, model_1_pred)
model_1_logloss = log_loss(y_test, model_1_prob)
model_1_brier = brier_score_loss(y_test, model_1_prob)

print("MODEL 1 — SRS + Home Court")
print("--------------------------")
print(f"Accuracy:    {model_1_accuracy:.3f}")
print(f"Log Loss:    {model_1_logloss:.3f}")
print(f"Brier Score: {model_1_brier:.3f}")

print("\nModel parameters:")
print(f"Intercept:       {model_1.intercept_[0]:.3f}")
print(f"SRS coefficient: {model_1.coef_[0][0]:.3f}")

MODEL 1 — SRS + Home Court
--------------------------
Accuracy:    0.643
Log Loss:    0.648
Brier Score: 0.227

Model parameters:
Intercept:       0.648
SRS coefficient: 0.158


In [8]:
# -----------------------------
# MODEL 2: OFFENSE + DEFENSE + HOME COURT
# -----------------------------

model_2_features = [
    "ORtg_Diff",
    "DRtg_Diff"
]

X_train_2 = train_data[model_2_features]
X_test_2 = test_data[model_2_features]

# Fit model
model_2 = LogisticRegression()
model_2.fit(X_train_2, y_train)

# Probabilities
model_2_prob = model_2.predict_proba(X_test_2)[:, 1]

# Winner predictions
model_2_pred = (model_2_prob >= 0.5).astype(int)

# Evaluate
model_2_accuracy = accuracy_score(y_test, model_2_pred)
model_2_logloss = log_loss(y_test, model_2_prob)
model_2_brier = brier_score_loss(y_test, model_2_prob)

print("MODEL COMPARISON")
print("----------------")

print("\nModel 1 — SRS + Home Court")
print(f"Accuracy:    {model_1_accuracy:.3f}")
print(f"Log Loss:    {model_1_logloss:.3f}")
print(f"Brier Score: {model_1_brier:.3f}")

print("\nModel 2 — Offense + Defense + Home Court")
print(f"Accuracy:    {model_2_accuracy:.3f}")
print(f"Log Loss:    {model_2_logloss:.3f}")
print(f"Brier Score: {model_2_brier:.3f}")

print("\nModel 2 coefficients:")
for feature, coefficient in zip(
    model_2_features,
    model_2.coef_[0]
):
    print(f"{feature}: {coefficient:.3f}")

print(f"Intercept: {model_2.intercept_[0]:.3f}")

MODEL COMPARISON
----------------

Model 1 — SRS + Home Court
Accuracy:    0.643
Log Loss:    0.648
Brier Score: 0.227

Model 2 — Offense + Defense + Home Court
Accuracy:    0.634
Log Loss:    0.649
Brier Score: 0.228

Model 2 coefficients:
ORtg_Diff: 0.133
DRtg_Diff: -0.147
Intercept: 0.647


In [9]:
team_data = pd.read_csv(
    "../data/nba_team_seasons_1986_2026.csv"
)

print("Team dataset loaded.")
print("Rows:", len(team_data))
print("Columns:", len(team_data.columns))

print("\nColumns:")
print(team_data.columns.tolist())

Team dataset loaded.
Rows: 1177
Columns: 31

Columns:
['Rk', 'Team', 'Age', 'W', 'L', 'PW', 'PL', 'MOV', 'SOS', 'SRS', 'ORtg', 'DRtg', 'NRtg', 'Pace', 'FTr', '3PAr', 'TS%', 'OFF_eFG%', 'OFF_TOV%', 'OFF_ORB%', 'OFF_FT/FGA', 'DEF_eFG%', 'DEF_TOV%', 'DEF_DRB%', 'DEF_FT/FGA', 'Arena', 'Attend.', 'Attend./G', 'Season_End', 'Season', 'Games']


In [10]:
# Calculate win percentage again in the loaded team dataset
team_data["W_PCT"] = (
    team_data["W"] /
    (team_data["W"] + team_data["L"])
)

expanded_features = [
    "Season",
    "Team",
    "W_PCT",
    "SRS",
    "ORtg",
    "DRtg",
    "NRtg",
    "Pace",
    "TS%",
    "OFF_eFG%",
    "OFF_TOV%",
    "OFF_ORB%",
    "OFF_FT/FGA",
    "DEF_eFG%",
    "DEF_TOV%",
    "DEF_DRB%",
    "DEF_FT/FGA"
]

team_expanded = team_data[expanded_features].copy()

print("Expanded team features:", len(team_expanded.columns))
print(team_expanded.columns.tolist())

Expanded team features: 17
['Season', 'Team', 'W_PCT', 'SRS', 'ORtg', 'DRtg', 'NRtg', 'Pace', 'TS%', 'OFF_eFG%', 'OFF_TOV%', 'OFF_ORB%', 'OFF_FT/FGA', 'DEF_eFG%', 'DEF_TOV%', 'DEF_DRB%', 'DEF_FT/FGA']


In [11]:
# Start again from the raw playoff-game information
expanded_model_data = model_data[
    [
        "Season",
        "Season_End",
        "Date",
        "Away_Team",
        "Away_PTS",
        "Home_Team",
        "Home_PTS",
        "Home_Win"
    ]
].copy()

# Variables to attach to both teams
stat_columns = [
    "W_PCT",
    "SRS",
    "ORtg",
    "DRtg",
    "NRtg",
    "Pace",
    "TS%",
    "OFF_eFG%",
    "OFF_TOV%",
    "OFF_ORB%",
    "OFF_FT/FGA",
    "DEF_eFG%",
    "DEF_TOV%",
    "DEF_DRB%",
    "DEF_FT/FGA"
]

# Create home-team version
home_expanded = team_expanded.rename(
    columns={
        "Team": "Home_Team",
        **{col: f"Home_{col}" for col in stat_columns}
    }
)

# Create away-team version
away_expanded = team_expanded.rename(
    columns={
        "Team": "Away_Team",
        **{col: f"Away_{col}" for col in stat_columns}
    }
)

# Merge home stats
expanded_model_data = expanded_model_data.merge(
    home_expanded,
    on=["Season", "Home_Team"],
    how="left"
)

# Merge away stats
expanded_model_data = expanded_model_data.merge(
    away_expanded,
    on=["Season", "Away_Team"],
    how="left"
)

print("Rows:", len(expanded_model_data))
print("Columns:", len(expanded_model_data.columns))

print("\nMissing home matches:")
print(expanded_model_data["Home_SRS"].isna().sum())

print("\nMissing away matches:")
print(expanded_model_data["Away_SRS"].isna().sum())

Rows: 3143
Columns: 38

Missing home matches:
0

Missing away matches:
0


In [12]:
# Create home-minus-away differences for every candidate feature

for col in stat_columns:
    expanded_model_data[f"{col}_Diff"] = (
        expanded_model_data[f"Home_{col}"] -
        expanded_model_data[f"Away_{col}"]
    )

diff_columns = [f"{col}_Diff" for col in stat_columns]

print("Difference features created:", len(diff_columns))
print(diff_columns)

print("\nMissing values:")
print(expanded_model_data[diff_columns].isna().sum())

Difference features created: 15
['W_PCT_Diff', 'SRS_Diff', 'ORtg_Diff', 'DRtg_Diff', 'NRtg_Diff', 'Pace_Diff', 'TS%_Diff', 'OFF_eFG%_Diff', 'OFF_TOV%_Diff', 'OFF_ORB%_Diff', 'OFF_FT/FGA_Diff', 'DEF_eFG%_Diff', 'DEF_TOV%_Diff', 'DEF_DRB%_Diff', 'DEF_FT/FGA_Diff']

Missing values:
W_PCT_Diff         0
SRS_Diff           0
ORtg_Diff          0
DRtg_Diff          0
NRtg_Diff          0
Pace_Diff          0
TS%_Diff           0
OFF_eFG%_Diff      0
OFF_TOV%_Diff      0
OFF_ORB%_Diff      0
OFF_FT/FGA_Diff    0
DEF_eFG%_Diff      0
DEF_TOV%_Diff      0
DEF_DRB%_Diff      0
DEF_FT/FGA_Diff    0
dtype: int64


In [13]:
# Correlations among candidate difference features

correlation_matrix = expanded_model_data[
    diff_columns
].corr()

# Show each feature's strongest correlation with another feature
for col in diff_columns:
    correlations = (
        correlation_matrix[col]
        .drop(col)
        .abs()
        .sort_values(ascending=False)
    )

    strongest_feature = correlations.index[0]
    strongest_value = correlations.iloc[0]

    print(
        f"{col:20s} -> "
        f"{strongest_feature:20s}: "
        f"{strongest_value:.3f}"
    )

W_PCT_Diff           -> NRtg_Diff           : 0.926
SRS_Diff             -> NRtg_Diff           : 0.996
ORtg_Diff            -> TS%_Diff            : 0.749
DRtg_Diff            -> DEF_eFG%_Diff       : 0.831
NRtg_Diff            -> SRS_Diff            : 0.996
Pace_Diff            -> OFF_eFG%_Diff       : 0.271
TS%_Diff             -> OFF_eFG%_Diff       : 0.947
OFF_eFG%_Diff        -> TS%_Diff            : 0.947
OFF_TOV%_Diff        -> ORtg_Diff           : 0.394
OFF_ORB%_Diff        -> OFF_eFG%_Diff       : 0.399
OFF_FT/FGA_Diff      -> TS%_Diff            : 0.298
DEF_eFG%_Diff        -> DRtg_Diff           : 0.831
DEF_TOV%_Diff        -> DEF_FT/FGA_Diff     : 0.450
DEF_DRB%_Diff        -> DRtg_Diff           : 0.459
DEF_FT/FGA_Diff      -> DEF_TOV%_Diff       : 0.450


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Model 3 tools loaded.")

Model 3 tools loaded.


In [15]:
# -----------------------------
# MODEL 3: SRS + FOUR FACTORS + PACE
# -----------------------------

model_3_features = [
    "SRS_Diff",
    "OFF_eFG%_Diff",
    "OFF_TOV%_Diff",
    "OFF_ORB%_Diff",
    "OFF_FT/FGA_Diff",
    "DEF_eFG%_Diff",
    "DEF_TOV%_Diff",
    "DEF_DRB%_Diff",
    "DEF_FT/FGA_Diff",
    "Pace_Diff"
]

# Same chronological split as before
train_3 = expanded_model_data[
    expanded_model_data["Season_End"] <= 2015
].copy()

test_3 = expanded_model_data[
    expanded_model_data["Season_End"] >= 2016
].copy()

X_train_3 = train_3[model_3_features]
y_train_3 = train_3["Home_Win"]

X_test_3 = test_3[model_3_features]
y_test_3 = test_3["Home_Win"]

# Standardize variables, then fit regularized logistic regression
model_3 = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression())
])

model_3.fit(X_train_3, y_train_3)

# Predicted home-win probabilities
model_3_prob = model_3.predict_proba(X_test_3)[:, 1]

# Winner predictions
model_3_pred = (model_3_prob >= 0.5).astype(int)

# Evaluate
model_3_accuracy = accuracy_score(y_test_3, model_3_pred)
model_3_logloss = log_loss(y_test_3, model_3_prob)
model_3_brier = brier_score_loss(y_test_3, model_3_prob)

print("MODEL COMPARISON")
print("----------------")

print("\nModel 1 — SRS + Home Court")
print(f"Accuracy:    {model_1_accuracy:.3f}")
print(f"Log Loss:    {model_1_logloss:.3f}")
print(f"Brier Score: {model_1_brier:.3f}")

print("\nModel 2 — Offense + Defense + Home Court")
print(f"Accuracy:    {model_2_accuracy:.3f}")
print(f"Log Loss:    {model_2_logloss:.3f}")
print(f"Brier Score: {model_2_brier:.3f}")

print("\nModel 3 — SRS + Four Factors + Pace")
print(f"Accuracy:    {model_3_accuracy:.3f}")
print(f"Log Loss:    {model_3_logloss:.3f}")
print(f"Brier Score: {model_3_brier:.3f}")

MODEL COMPARISON
----------------

Model 1 — SRS + Home Court
Accuracy:    0.643
Log Loss:    0.648
Brier Score: 0.227

Model 2 — Offense + Defense + Home Court
Accuracy:    0.634
Log Loss:    0.649
Brier Score: 0.228

Model 3 — SRS + Four Factors + Pace
Accuracy:    0.643
Log Loss:    0.650
Brier Score: 0.227


In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier

print("Gradient boosting loaded.")

Gradient boosting loaded.


In [17]:
# -----------------------------
# MODEL 4: GRADIENT BOOSTING
# -----------------------------

model_4_features = [
    "SRS_Diff",
    "OFF_eFG%_Diff",
    "OFF_TOV%_Diff",
    "OFF_ORB%_Diff",
    "OFF_FT/FGA_Diff",
    "DEF_eFG%_Diff",
    "DEF_TOV%_Diff",
    "DEF_DRB%_Diff",
    "DEF_FT/FGA_Diff",
    "Pace_Diff"
]

X_train_4 = train_3[model_4_features]
X_test_4 = test_3[model_4_features]

# Conservative settings to reduce overfitting
model_4 = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=100,
    max_leaf_nodes=7,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=42
)

model_4.fit(X_train_4, y_train_3)

# Predicted probabilities
model_4_prob = model_4.predict_proba(X_test_4)[:, 1]

# Winner predictions
model_4_pred = (model_4_prob >= 0.5).astype(int)

# Evaluate
model_4_accuracy = accuracy_score(y_test_3, model_4_pred)
model_4_logloss = log_loss(y_test_3, model_4_prob)
model_4_brier = brier_score_loss(y_test_3, model_4_prob)

print("MODEL COMPARISON")
print("----------------")

print("\nModel 1 — SRS + Home Court")
print(f"Accuracy:    {model_1_accuracy:.3f}")
print(f"Log Loss:    {model_1_logloss:.3f}")
print(f"Brier Score: {model_1_brier:.3f}")

print("\nModel 3 — SRS + Four Factors + Pace")
print(f"Accuracy:    {model_3_accuracy:.3f}")
print(f"Log Loss:    {model_3_logloss:.3f}")
print(f"Brier Score: {model_3_brier:.3f}")

print("\nModel 4 — Gradient Boosting")
print(f"Accuracy:    {model_4_accuracy:.3f}")
print(f"Log Loss:    {model_4_logloss:.3f}")
print(f"Brier Score: {model_4_brier:.3f}")

MODEL COMPARISON
----------------

Model 1 — SRS + Home Court
Accuracy:    0.643
Log Loss:    0.648
Brier Score: 0.227

Model 3 — SRS + Four Factors + Pace
Accuracy:    0.643
Log Loss:    0.650
Brier Score: 0.227

Model 4 — Gradient Boosting
Accuracy:    0.646
Log Loss:    0.659
Brier Score: 0.232


In [18]:
# -----------------------------
# ROLLING VALIDATION — MODEL 1
# -----------------------------

rolling_predictions_1 = []

# Start predicting with the 2000-01 postseason.
# Each year, train only on seasons that occurred before it.
for test_year in range(2001, 2026):

    rolling_train = model_data[
        model_data["Season_End"] < test_year
    ].copy()

    rolling_test = model_data[
        model_data["Season_End"] == test_year
    ].copy()

    # Skip if a season somehow has no games
    if len(rolling_test) == 0:
        continue

    X_roll_train = rolling_train[["SRS_Diff"]]
    y_roll_train = rolling_train["Home_Win"]

    X_roll_test = rolling_test[["SRS_Diff"]]

    roll_model = LogisticRegression()
    roll_model.fit(X_roll_train, y_roll_train)

    roll_prob = roll_model.predict_proba(X_roll_test)[:, 1]

    season_results = rolling_test[
        [
            "Season",
            "Season_End",
            "Home_Win"
        ]
    ].copy()

    season_results["Predicted_Prob"] = roll_prob

    rolling_predictions_1.append(season_results)

# Combine all out-of-sample predictions
rolling_results_1 = pd.concat(
    rolling_predictions_1,
    ignore_index=True
)

rolling_results_1["Predicted_Win"] = (
    rolling_results_1["Predicted_Prob"] >= 0.5
).astype(int)

# Overall rolling-validation metrics
rolling_1_accuracy = accuracy_score(
    rolling_results_1["Home_Win"],
    rolling_results_1["Predicted_Win"]
)

rolling_1_logloss = log_loss(
    rolling_results_1["Home_Win"],
    rolling_results_1["Predicted_Prob"]
)

rolling_1_brier = brier_score_loss(
    rolling_results_1["Home_Win"],
    rolling_results_1["Predicted_Prob"]
)

print("ROLLING VALIDATION — MODEL 1")
print("----------------------------")
print("Test seasons:",
      rolling_results_1["Season"].nunique())
print("Out-of-sample games:",
      len(rolling_results_1))

print(f"\nAccuracy:    {rolling_1_accuracy:.3f}")
print(f"Log Loss:    {rolling_1_logloss:.3f}")
print(f"Brier Score: {rolling_1_brier:.3f}")

ROLLING VALIDATION — MODEL 1
----------------------------
Test seasons: 25
Out-of-sample games: 2071

Accuracy:    0.661
Log Loss:    0.628
Brier Score: 0.219


In [19]:
def rolling_validate(model_number):

    all_predictions = []

    for test_year in range(2001, 2026):

        if model_number in [1, 2]:
            data = model_data
        else:
            data = expanded_model_data

        train = data[
            data["Season_End"] < test_year
        ].copy()

        test = data[
            data["Season_End"] == test_year
        ].copy()

        if len(test) == 0:
            continue

        y_train = train["Home_Win"]
        y_test = test["Home_Win"]

        # MODEL 1
        if model_number == 1:

            features = ["SRS_Diff"]

            model = LogisticRegression()

        # MODEL 2
        elif model_number == 2:

            features = [
                "ORtg_Diff",
                "DRtg_Diff"
            ]

            model = LogisticRegression()

        # MODEL 3
        elif model_number == 3:

            features = model_3_features

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("logistic", LogisticRegression())
            ])

        # MODEL 4
        elif model_number == 4:

            features = model_4_features

            model = HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=100,
                max_leaf_nodes=7,
                min_samples_leaf=30,
                l2_regularization=1.0,
                random_state=42
            )

        X_train = train[features]
        X_test = test[features]

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        results = test[
            ["Season", "Season_End", "Home_Win"]
        ].copy()

        results["Predicted_Prob"] = probabilities

        all_predictions.append(results)

    results = pd.concat(
        all_predictions,
        ignore_index=True
    )

    results["Predicted_Win"] = (
        results["Predicted_Prob"] >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        results["Home_Win"],
        results["Predicted_Win"]
    )

    loss = log_loss(
        results["Home_Win"],
        results["Predicted_Prob"]
    )

    brier = brier_score_loss(
        results["Home_Win"],
        results["Predicted_Prob"]
    )

    return accuracy, loss, brier


# Run rolling validation for all four models
rolling_comparison = []

for model_number in [1, 2, 3, 4]:

    accuracy, loss, brier = rolling_validate(
        model_number
    )

    rolling_comparison.append({
        "Model": model_number,
        "Accuracy": accuracy,
        "Log_Loss": loss,
        "Brier": brier
    })

rolling_comparison = pd.DataFrame(
    rolling_comparison
)

print(
    rolling_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

 Model  Accuracy  Log_Loss  Brier
     1     0.661     0.628  0.219
     2     0.656     0.629  0.219
     3     0.656     0.631  0.220
     4     0.653     0.640  0.224


In [20]:
# -----------------------------
# CALIBRATION CHECK — MODEL 1
# -----------------------------

# Use the rolling out-of-sample predictions
calibration_data = rolling_results_1.copy()

# Put predictions into probability bins
calibration_data["Probability_Bin"] = pd.cut(
    calibration_data["Predicted_Prob"],
    bins=[
        0.0,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.9,
        1.0
    ],
    include_lowest=True
)

calibration_table = (
    calibration_data
    .groupby(
        "Probability_Bin",
        observed=True
    )
    .agg(
        Games=("Home_Win", "size"),
        Avg_Predicted_Prob=("Predicted_Prob", "mean"),
        Actual_Home_Win_Rate=("Home_Win", "mean")
    )
    .reset_index()
)

calibration_table["Difference"] = (
    calibration_table["Actual_Home_Win_Rate"] -
    calibration_table["Avg_Predicted_Prob"]
)

calibration_table

,Probability_Bin,Games,Avg_Predicted_Prob,Actual_Home_Win_Rate,Difference
0,"(-0.001, 0.4]",108,0.338505,0.351852,0.013347
1,"(0.4, 0.5]",174,0.463884,0.356322,-0.107562
2,"(0.5, 0.6]",360,0.549077,0.525000,-0.024077
3,"(0.6, 0.7]",604,0.651965,0.627483,-0.024482
4,"(0.7, 0.8]",565,0.749079,0.725664,-0.023415
5,"(0.8, 0.9]",238,0.841048,0.794118,-0.046930
6,"(0.9, 1.0]",22,0.914533,0.909091,-0.005442


In [21]:
# -----------------------------
# HOME-COURT ADVANTAGE BY ERA
# -----------------------------

era_home_court = (
    model_data
    .assign(
        Era=pd.cut(
            model_data["Season_End"],
            bins=[1985, 1995, 2005, 2015, 2025],
            labels=[
                "1985-86 to 1994-95",
                "1995-96 to 2004-05",
                "2005-06 to 2014-15",
                "2015-16 to 2024-25"
            ]
        )
    )
    .groupby("Era", observed=True)
    .agg(
        Games=("Home_Win", "size"),
        Home_Wins=("Home_Win", "sum"),
        Home_Win_Rate=("Home_Win", "mean")
    )
    .reset_index()
)

era_home_court

,Era,Games,Home_Wins,Home_Win_Rate
0,1985-86 to 1994-95,720,483,0.670833
1,1995-96 to 2004-05,748,480,0.641711
2,2005-06 to 2014-15,841,551,0.655172
3,2015-16 to 2024-25,834,493,0.591127


In [22]:
# ---------------------------------------
# HOME-COURT EFFECT BY ERA
# CONTROLLING FOR TEAM STRENGTH (SRS)
# ---------------------------------------

era_definitions = [
    ("1985-86 to 1994-95", 1986, 1995),
    ("1995-96 to 2004-05", 1996, 2005),
    ("2005-06 to 2014-15", 2006, 2015),
    ("2015-16 to 2024-25", 2016, 2025)
]

era_results = []

for era_name, start_year, end_year in era_definitions:

    era_data = model_data[
        (model_data["Season_End"] >= start_year) &
        (model_data["Season_End"] <= end_year)
    ].copy()

    X_era = era_data[["SRS_Diff"]]
    y_era = era_data["Home_Win"]

    era_model = LogisticRegression()
    era_model.fit(X_era, y_era)

    intercept = era_model.intercept_[0]
    srs_coef = era_model.coef_[0][0]

    # Estimated probability that the home team wins
    # when the teams have identical SRS
    equal_team_home_prob = 1 / (1 + np.exp(-intercept))

    era_results.append({
        "Era": era_name,
        "Games": len(era_data),
        "Intercept": intercept,
        "SRS_Coefficient": srs_coef,
        "Equal_Team_Home_Win_Prob": equal_team_home_prob
    })

era_results = pd.DataFrame(era_results)

era_results

,Era,Games,Intercept,SRS_Coefficient,Equal_Team_Home_Win_Prob
0,1985-86 to 1994-95,720,0.698425,0.135470,0.667839
1,1995-96 to 2004-05,748,0.590498,0.202952,0.643479
2,2005-06 to 2014-15,841,0.657569,0.146965,0.658714
3,2015-16 to 2024-25,834,0.353249,0.136029,0.587405


In [23]:
# ---------------------------------------
# TEST DIFFERENT HISTORICAL TRAINING WINDOWS
# ---------------------------------------

window_results = []

for window in [10, 15, 20, None]:

    all_predictions = []

    for test_year in range(2001, 2026):

        # Use only seasons BEFORE the test season
        if window is None:
            train = model_data[
                model_data["Season_End"] < test_year
            ].copy()

            window_name = "All Prior"

        else:
            train = model_data[
                (model_data["Season_End"] < test_year) &
                (model_data["Season_End"] >= test_year - window)
            ].copy()

            window_name = f"{window} Years"

        test = model_data[
            model_data["Season_End"] == test_year
        ].copy()

        if len(test) == 0:
            continue

        X_train = train[["SRS_Diff"]]
        y_train = train["Home_Win"]

        X_test = test[["SRS_Diff"]]

        model = LogisticRegression()
        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        results = test[
            ["Season", "Season_End", "Home_Win"]
        ].copy()

        results["Predicted_Prob"] = probabilities

        all_predictions.append(results)

    combined = pd.concat(
        all_predictions,
        ignore_index=True
    )

    combined["Predicted_Win"] = (
        combined["Predicted_Prob"] >= 0.5
    ).astype(int)

    window_results.append({
        "Training_Window": window_name,

        "Accuracy": accuracy_score(
            combined["Home_Win"],
            combined["Predicted_Win"]
        ),

        "Log_Loss": log_loss(
            combined["Home_Win"],
            combined["Predicted_Prob"]
        ),

        "Brier": brier_score_loss(
            combined["Home_Win"],
            combined["Predicted_Prob"]
        )
    })

window_results = pd.DataFrame(window_results)

print(
    window_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

Training_Window  Accuracy  Log_Loss  Brier
       10 Years     0.671     0.627  0.218
       15 Years     0.665     0.627  0.218
       20 Years     0.665     0.627  0.218
      All Prior     0.661     0.628  0.219


In [24]:
# ---------------------------------------
# SEARCH FOR BEST TRAINING WINDOW
# ---------------------------------------

window_search_results = []

for window in range(5, 26):

    all_predictions = []

    for test_year in range(2001, 2026):

        train = model_data[
            (model_data["Season_End"] < test_year) &
            (model_data["Season_End"] >= test_year - window)
        ].copy()

        test = model_data[
            model_data["Season_End"] == test_year
        ].copy()

        if len(test) == 0:
            continue

        X_train = train[["SRS_Diff"]]
        y_train = train["Home_Win"]
        X_test = test[["SRS_Diff"]]

        model = LogisticRegression()
        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        results = test[
            ["Season", "Season_End", "Home_Win"]
        ].copy()

        results["Predicted_Prob"] = probabilities
        all_predictions.append(results)

    combined = pd.concat(
        all_predictions,
        ignore_index=True
    )

    combined["Predicted_Win"] = (
        combined["Predicted_Prob"] >= 0.5
    ).astype(int)

    window_search_results.append({
        "Years": window,
        "Accuracy": accuracy_score(
            combined["Home_Win"],
            combined["Predicted_Win"]
        ),
        "Log_Loss": log_loss(
            combined["Home_Win"],
            combined["Predicted_Prob"]
        ),
        "Brier": brier_score_loss(
            combined["Home_Win"],
            combined["Predicted_Prob"]
        )
    })

window_search_results = pd.DataFrame(
    window_search_results
)

window_search_results.sort_values(
    ["Log_Loss", "Brier"]
).head(10)

,Years,Accuracy,Log_Loss,Brier
2,7,0.662482,0.626152,0.217946
3,8,0.662482,0.626267,0.218002
0,5,0.660068,0.626395,0.218034
1,6,0.663930,0.626472,0.218051
5,10,0.670690,0.626753,0.218203
14,19,0.664896,0.626920,0.218250
15,20,0.665379,0.626923,0.218251
13,18,0.664413,0.626970,0.218270
11,16,0.660550,0.626980,0.218281
9,14,0.662965,0.626985,0.218339


In [25]:
# ---------------------------------------
# FINAL HOLDOUT TEST: 2020-21 TO 2024-25
# ---------------------------------------

def final_holdout_test(window):

    predictions = []

    for test_year in range(2021, 2026):

        if window is None:
            train = model_data[
                model_data["Season_End"] < test_year
            ].copy()
            name = "All Prior"

        else:
            train = model_data[
                (model_data["Season_End"] < test_year) &
                (model_data["Season_End"] >= test_year - window)
            ].copy()
            name = f"{window} Years"

        test = model_data[
            model_data["Season_End"] == test_year
        ].copy()

        X_train = train[["SRS_Diff"]]
        y_train = train["Home_Win"]
        X_test = test[["SRS_Diff"]]

        model = LogisticRegression()
        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        results = test[
            ["Season", "Season_End", "Home_Win"]
        ].copy()

        results["Predicted_Prob"] = probabilities

        predictions.append(results)

    combined = pd.concat(
        predictions,
        ignore_index=True
    )

    combined["Predicted_Win"] = (
        combined["Predicted_Prob"] >= 0.5
    ).astype(int)

    return {
        "Model": name,
        "Games": len(combined),
        "Accuracy": accuracy_score(
            combined["Home_Win"],
            combined["Predicted_Win"]
        ),
        "Log_Loss": log_loss(
            combined["Home_Win"],
            combined["Predicted_Prob"]
        ),
        "Brier": brier_score_loss(
            combined["Home_Win"],
            combined["Predicted_Prob"]
        )
    }


final_results = pd.DataFrame([
    final_holdout_test(7),
    final_holdout_test(10),
    final_holdout_test(None)
])

print(
    final_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

    Model  Games  Accuracy  Log_Loss  Brier
  7 Years    422     0.640     0.655  0.231
 10 Years    422     0.647     0.657  0.231
All Prior    422     0.611     0.664  0.234


In [26]:
# ---------------------------------------
# 7-YEAR ROLLING MODEL — CALIBRATION
# ---------------------------------------

rolling_predictions_7yr = []

for test_year in range(2001, 2026):

    # Train only on the seven seasons immediately
    # preceding the postseason being predicted
    train = model_data[
        (model_data["Season_End"] < test_year) &
        (model_data["Season_End"] >= test_year - 7)
    ].copy()

    test = model_data[
        model_data["Season_End"] == test_year
    ].copy()

    if len(test) == 0:
        continue

    X_train = train[["SRS_Diff"]]
    y_train = train["Home_Win"]

    X_test = test[["SRS_Diff"]]

    model = LogisticRegression()
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    results = test[
        ["Season", "Season_End", "Home_Win"]
    ].copy()

    results["Predicted_Prob"] = probabilities

    rolling_predictions_7yr.append(results)

# Combine all out-of-sample predictions
rolling_results_7yr = pd.concat(
    rolling_predictions_7yr,
    ignore_index=True
)

# Create probability bins
rolling_results_7yr["Probability_Bin"] = pd.cut(
    rolling_results_7yr["Predicted_Prob"],
    bins=[
        0.0,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.9,
        1.0
    ],
    include_lowest=True
)

# Calibration table
calibration_7yr = (
    rolling_results_7yr
    .groupby(
        "Probability_Bin",
        observed=True
    )
    .agg(
        Games=("Home_Win", "size"),
        Avg_Predicted_Prob=("Predicted_Prob", "mean"),
        Actual_Home_Win_Rate=("Home_Win", "mean")
    )
    .reset_index()
)

calibration_7yr["Difference"] = (
    calibration_7yr["Actual_Home_Win_Rate"] -
    calibration_7yr["Avg_Predicted_Prob"]
)

calibration_7yr

,Probability_Bin,Games,Avg_Predicted_Prob,Actual_Home_Win_Rate,Difference
0,"(-0.001, 0.4]",134,0.318981,0.343284,0.024302
1,"(0.4, 0.5]",243,0.458649,0.411523,-0.047127
2,"(0.5, 0.6]",384,0.552395,0.572917,0.020522
3,"(0.6, 0.7]",559,0.650262,0.645796,-0.004466
4,"(0.7, 0.8]",490,0.745149,0.712245,-0.032905
5,"(0.8, 0.9]",247,0.837736,0.797571,-0.040165
6,"(0.9, 1.0]",14,0.925361,1.000000,0.074639
